# 10 DINO Patch Matching Probe

역할: mask, prototype, OT를 모두 빼고 DINO patch feature만으로 clean image와 shifted image가 patch level에서 어떻게 대응되는지 확인한다.

확인할 것:
- 같은 위치 patch의 cosine similarity가 shift에서 얼마나 떨어지는가
- query patch의 nearest clean patch가 같은 위치인지, 주변 위치인지, 멀리 이동하는지
- `delta_i = f_shift_i - f_clean_i`가 representation space에서 비슷한 방향/low-dimensional subspace로 움직이는지
- brightness / blur / noise / position shift별 displacement와 feature-shift 패턴


## Cell Role: Repo Setup

Colab에서는 `/content/ReGraM` checkout을 사용하고, 로컬에서는 현재 repo root를 찾는다. 이 노트북은 raw LOCO image와 DINO만 사용하며 SAM/GroundingDINO mask를 읽지 않는다.


In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import math
import os
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_URL = 'https://github.com/outSeop/ReGraM.git'


def is_regram_root(path: Path) -> bool:
    return (
        (path / 'experiments' / 'validation' / 'condition_shift_baseline').exists()
        and (path / 'manifests').exists()
    )


colab_checkout = Path('/content/ReGraM')
if Path('/content').exists():
    if (colab_checkout / '.git').exists():
        subprocess.run(['git', '-C', str(colab_checkout), 'fetch', 'origin', 'main'], check=True)
        subprocess.run(
            [
                'git', '-C', str(colab_checkout), 'checkout', 'FETCH_HEAD', '--',
                'experiments/validation/condition_shift_baseline/src',
                'experiments/validation/condition_shift_baseline/notebook/10_dino_patch_matching_probe.ipynb',
            ],
            check=True,
        )
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(colab_checkout)], check=True)

cwd = Path.cwd().resolve()
repo_candidates = [colab_checkout, cwd, *cwd.parents]
REPO_ROOT = next((candidate.resolve() for candidate in repo_candidates if candidate.exists() and is_regram_root(candidate)), None)
if REPO_ROOT is None:
    raise FileNotFoundError('ReGraM checkout not found')

EXP_ROOT = REPO_ROOT / 'experiments' / 'validation' / 'condition_shift_baseline'
SRC_ROOT = EXP_ROOT / 'src'
for import_root in [SRC_ROOT, SRC_ROOT / 'orchestration']:
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

print('REPO_ROOT =', REPO_ROOT)
print('EXP_ROOT =', EXP_ROOT)


## Cell Role: Dataset And Shift Setup

`test/good` clean image 몇 장을 기준으로 high severity shift image를 만든다. 이미 06/09에서 생성된 shifted image가 있으면 재사용하고, 없으면 augmentation만 적용해서 저장한다.


In [ ]:
from PIL import Image

from data.augmentation_runtime import DEFAULT_PARAMS, apply_augmentation

CATEGORY = 'breakfast_box'
SPLIT = 'test'
SAMPLE_IDS = ['000.png', '001.png', '002.png']
SHIFT_CONDITIONS = [
    ('clean_identity', 'identity', 'none'),
    ('brightness_high', 'brightness', 'high'),
    ('gaussian_blur_high', 'gaussian_blur', 'high'),
    ('gaussian_noise_high', 'gaussian_noise', 'high'),
    ('position_shift_high', 'position_shift', 'high'),
]
FEATURE_BACKBONE = 'dinov2_vits14'
IMAGE_SIZE = 224

RAW_LOCO_ROW_ROOT = REPO_ROOT / 'data' / 'row'
RAW_LOCO_ROOT = RAW_LOCO_ROW_ROOT / 'mvtec_loco_anomaly_detection'
PROBE_ROOT = EXP_ROOT / 'reports' / 'dino_patch_matching_probe'
SHIFTED_IMAGE_ROOT = PROBE_ROOT / 'images'
SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_dino_patch_match_summary.csv'
PROBE_ROOT.mkdir(parents=True, exist_ok=True)

DRIVE_RAW_TAR = Path('/content/drive/MyDrive/ReGraM/data/row/mvtec_loco_anomaly_detection.tar.gz')


def mount_drive_if_available() -> None:
    if not Path('/content').exists() or Path('/content/drive/MyDrive').exists():
        return
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print(f'Google Drive mount skipped: {type(exc).__name__}: {exc}')


def normalize_dataset_root(parent: Path, dataset_name: str) -> Path:
    direct = parent / dataset_name
    nested_candidates = [parent / 'content' / dataset_name, parent / 'data' / 'row' / dataset_name]
    if direct.exists():
        return direct
    for candidate in nested_candidates:
        if candidate.exists():
            direct.parent.mkdir(parents=True, exist_ok=True)
            print(f'normalize dataset root: {candidate} -> {direct}')
            shutil.move(str(candidate), str(direct))
            return direct
    return direct


def restore_raw_loco_from_drive_if_needed() -> None:
    global RAW_LOCO_ROOT
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')
    if (RAW_LOCO_ROOT / CATEGORY / SPLIT / 'good').exists():
        return
    mount_drive_if_available()
    if not DRIVE_RAW_TAR.exists():
        print(f'raw LOCO Drive tar not found: {DRIVE_RAW_TAR}')
        return
    RAW_LOCO_ROW_ROOT.mkdir(parents=True, exist_ok=True)
    print(f'extract raw LOCO from Drive: {DRIVE_RAW_TAR} -> {RAW_LOCO_ROW_ROOT}')
    subprocess.run(['tar', '-xf', str(DRIVE_RAW_TAR), '-C', str(RAW_LOCO_ROW_ROOT)], check=True)
    RAW_LOCO_ROOT = normalize_dataset_root(RAW_LOCO_ROW_ROOT, 'mvtec_loco_anomaly_detection')


def clean_image_path(image_id: str) -> Path:
    return RAW_LOCO_ROOT / CATEGORY / SPLIT / 'good' / image_id


def shifted_image_path(image_id: str, condition: str) -> Path:
    return SHIFTED_IMAGE_ROOT / CATEGORY / condition / image_id


def stable_seed(image_id: str, condition: str) -> int:
    return 20260520 + sum(ord(ch) for ch in f'{CATEGORY}/{image_id}/{condition}')


restore_raw_loco_from_drive_if_needed()
missing = [path for path in [clean_image_path(image_id) for image_id in SAMPLE_IDS] if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing clean images: {missing}')

query_rows = []
for image_id in SAMPLE_IDS:
    source_path = clean_image_path(image_id)
    source_image = Image.open(source_path).convert('RGB')
    for condition, augmentation_type, severity in SHIFT_CONDITIONS:
        if condition == 'clean_identity':
            query_path = source_path
        else:
            query_path = shifted_image_path(image_id, condition)
            if not query_path.exists():
                query_path.parent.mkdir(parents=True, exist_ok=True)
                shifted = apply_augmentation(
                    source_image,
                    augmentation_type=augmentation_type,
                    severity=severity,
                    seed=stable_seed(image_id, condition),
                    params=DEFAULT_PARAMS[augmentation_type][severity],
                )
                shifted.save(query_path)
        query_rows.append({
            'image_id': image_id,
            'condition': condition,
            'augmentation_type': augmentation_type,
            'severity': severity,
            'reference_path': str(source_path),
            'query_path': str(query_path),
        })

query_df = pd.DataFrame(query_rows)
display(query_df)


## Cell Role: DINO Setup

DINOv2 patch feature extractor를 준비한다. 이후 모든 비교는 DINO patch feature만 사용한다.


In [ ]:
import numpy as np
import torch

from stage1_adapter import (
    analyze_dino_representation_shift,
    match_dino_patch_grid,
    summarize_dino_patch_match,
    summarize_dino_representation_shift,
)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

dino_model = torch.hub.load('facebookresearch/dinov2', FEATURE_BACKBONE).to(device).eval()
dino_mean = torch.tensor([0.485, 0.456, 0.406], dtype=torch.float32, device=device).view(1, 3, 1, 1)
dino_std = torch.tensor([0.229, 0.224, 0.225], dtype=torch.float32, device=device).view(1, 3, 1, 1)


def preprocess_dino(image: Image.Image) -> torch.Tensor:
    image = image.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), resample=Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array).permute(2, 0, 1).unsqueeze(0).to(device)
    return (tensor - dino_mean) / dino_std


@torch.no_grad()
def extract_dino_feature_map(image_path: Path) -> np.ndarray:
    image = Image.open(image_path).convert('RGB')
    output = dino_model.forward_features(preprocess_dino(image))
    tokens = output['x_norm_patchtokens'] if isinstance(output, dict) else output
    tokens = tokens.squeeze(0).detach().cpu().float()
    grid = int(math.sqrt(tokens.shape[0]))
    if grid * grid != tokens.shape[0]:
        raise ValueError(f'DINO token count is not square: {tokens.shape[0]}')
    return tokens.reshape(grid, grid, tokens.shape[-1]).numpy()

print('DINO backbone ready:', FEATURE_BACKBONE)


## Cell Role: Run DINO-Only Patch Matching

각 query image의 DINO patch를 같은 image의 clean DINO patch와 top-1 nearest-neighbor matching한다. 저장되는 metric은 같은 좌표 similarity, top-1 similarity, top-1이 같은 위치인지, top-1 displacement가 얼마나 큰지다.


In [ ]:
feature_cache = {}
match_results = {}
summary_rows = []


def cached_feature_map(path: Path) -> np.ndarray:
    key = str(path)
    if key not in feature_cache:
        feature_cache[key] = extract_dino_feature_map(path)
    return feature_cache[key]


for _, row in query_df.iterrows():
    reference_path = Path(row['reference_path'])
    query_path = Path(row['query_path'])
    reference_feature_map = cached_feature_map(reference_path)
    query_feature_map = cached_feature_map(query_path)
    result = match_dino_patch_grid(reference_feature_map, query_feature_map)
    key = (row['image_id'], row['condition'])
    match_results[key] = result
    summary = summarize_dino_patch_match(result)
    summary.update(row.to_dict())
    summary_rows.append(summary)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)
print(f'saved summary: {SUMMARY_CSV}')
display(summary_df)

condition_summary_df = summary_df.groupby('condition').agg(
    n=('image_id', 'count'),
    identity_similarity_mean=('identity_similarity_mean', 'mean'),
    top1_similarity_mean=('top1_similarity_mean', 'mean'),
    identity_top1_ratio=('identity_top1_ratio', 'mean'),
    local_3x3_top1_ratio=('local_3x3_top1_ratio', 'mean'),
    mean_displacement_patches=('mean_displacement_patches', 'mean'),
    p90_displacement_patches=('p90_displacement_patches', 'mean'),
).reset_index()
display(condition_summary_df)


## Cell Role: Visualize Patch Matching Maps

왼쪽부터 clean reference, query image, 같은 좌표 cosine similarity, top-1 cosine similarity, top-1 displacement magnitude를 본다. displacement가 커지면 DINO feature가 같은 물리 위치보다 다른 clean patch를 더 가깝게 본다는 뜻이다.


In [ ]:
import matplotlib.pyplot as plt

VIS_SAMPLE_IDS = SAMPLE_IDS[:1]
VIS_CONDITIONS = [condition for condition, _, _ in SHIFT_CONDITIONS]


def show_match_row(image_id: str, condition: str) -> None:
    row = query_df[(query_df['image_id'].eq(image_id)) & (query_df['condition'].eq(condition))].iloc[0]
    result = match_results[(image_id, condition)]
    reference_image = np.asarray(Image.open(row['reference_path']).convert('RGB'))
    query_image = np.asarray(Image.open(row['query_path']).convert('RGB'))
    fig, axes = plt.subplots(1, 5, figsize=(18, 3.8))
    panels = [
        (reference_image, f'clean reference\n{image_id}', None),
        (query_image, f'query\n{condition}', None),
        (result.identity_similarity_map, 'same-location cosine', 'viridis'),
        (result.top1_similarity_map, 'top-1 cosine', 'viridis'),
        (result.displacement_norm_map, 'top-1 displacement\n(patches)', 'magma'),
    ]
    for ax, (image, title, cmap) in zip(axes, panels, strict=True):
        im = ax.imshow(image, cmap=cmap, vmin=0 if cmap else None)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
        if cmap:
            fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    metrics = result.metrics
    fig.suptitle(
        ' | '.join([
            f"identity_cos={metrics['identity_similarity_mean']:.3f}",
            f"top1_cos={metrics['top1_similarity_mean']:.3f}",
            f"same_pos={metrics['identity_top1_ratio']:.2f}",
            f"disp={metrics['mean_displacement_patches']:.2f}",
        ]),
        y=1.02,
        fontsize=10,
    )
    plt.tight_layout()
    plt.show()


for image_id in VIS_SAMPLE_IDS:
    for condition in VIS_CONDITIONS:
        show_match_row(image_id, condition)


## Cell Role: Representation Shift Diagnostics

같은 위치 patch 기준으로 `delta = shifted_feature - clean_feature`를 만들고, feature displacement가 전역적으로 비슷한 방향인지 또는 low-dimensional subspace에 놓이는지 본다. PCA는 두 가지를 본다.

- combined PCA: clean/shifted patch feature를 같은 PCA 공간에 놓고 clean→shift arrow를 그림
- delta PCA: `delta` vector 자체의 분포와 explained variance를 봄


In [ ]:
shift_results = {}
shift_rows = []

for _, row in query_df.iterrows():
    reference_path = Path(row['reference_path'])
    query_path = Path(row['query_path'])
    reference_feature_map = cached_feature_map(reference_path)
    query_feature_map = cached_feature_map(query_path)
    result = analyze_dino_representation_shift(reference_feature_map, query_feature_map)
    key = (row['image_id'], row['condition'])
    shift_results[key] = result
    summary = summarize_dino_representation_shift(result)
    summary.update(row.to_dict())
    shift_rows.append(summary)

shift_df = pd.DataFrame(shift_rows)
SHIFT_SUMMARY_CSV = PROBE_ROOT / f'{CATEGORY}_dino_representation_shift_summary.csv'
shift_df.to_csv(SHIFT_SUMMARY_CSV, index=False)
print(f'saved representation shift summary: {SHIFT_SUMMARY_CSV}')
display(shift_df)

shift_condition_summary_df = shift_df.groupby('condition').agg(
    n=('image_id', 'count'),
    mean_delta_norm=('mean_delta_norm', 'mean'),
    mean_delta_vector_norm=('mean_delta_vector_norm', 'mean'),
    direction_consistency=('delta_direction_consistency_mean', 'mean'),
    pairwise_delta_cosine=('pairwise_delta_cosine_mean', 'mean'),
    residual_after_mean_shift=('residual_after_mean_shift_mean', 'mean'),
    delta_pc1_explained=('delta_pc1_explained', 'mean'),
    delta_pc3_cumulative=('delta_pc3_cumulative', 'mean'),
).reset_index()
display(shift_condition_summary_df)


## Cell Role: PCA Arrow Plot

DINO feature를 PCA 2D에 투영한 뒤, 같은 patch의 clean 위치에서 shifted 위치로 arrow를 그린다. 전역 nuisance direction이 강하면 arrow가 비슷한 방향으로 정렬된다. patch별 반응이 다르면 arrow 방향이 흩어진다.


In [ ]:
PCA_SAMPLE_ID = SAMPLE_IDS[0]
PCA_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']
PCA_ARROW_STRIDE = 2

fig, axes = plt.subplots(1, len(PCA_CONDITIONS), figsize=(5 * len(PCA_CONDITIONS), 4.5))
if len(PCA_CONDITIONS) == 1:
    axes = [axes]

for ax, condition in zip(axes, PCA_CONDITIONS, strict=True):
    result = shift_results[(PCA_SAMPLE_ID, condition)]
    clean_xy = result.clean_pca_xy
    query_xy = result.query_pca_xy
    indices = np.arange(len(clean_xy))[::PCA_ARROW_STRIDE]
    ax.scatter(clean_xy[:, 0], clean_xy[:, 1], s=14, alpha=0.45, label='clean')
    ax.scatter(query_xy[:, 0], query_xy[:, 1], s=14, alpha=0.45, label='shifted')
    ax.quiver(
        clean_xy[indices, 0],
        clean_xy[indices, 1],
        query_xy[indices, 0] - clean_xy[indices, 0],
        query_xy[indices, 1] - clean_xy[indices, 1],
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.004,
        alpha=0.65,
    )
    ax.set_title(
        f"{condition}
"
        f"dir={result.metrics['delta_direction_consistency_mean']:.2f} "
        f"pc1={result.metrics['delta_pc1_explained']:.2f}"
    )
    ax.set_xlabel('PCA-1')
    ax.set_ylabel('PCA-2')
    ax.grid(True, alpha=0.2)
axes[0].legend(loc='best')
plt.tight_layout()
plt.show()


## Cell Role: Delta PCA And Delta Maps

`delta` vector만 PCA에 올려 condition별 분포를 본다. 오른쪽 map들은 patch별 delta norm과 평균 delta 방향과의 cosine similarity다.


In [ ]:
DELTA_SAMPLE_ID = SAMPLE_IDS[0]
DELTA_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']

fig, axes = plt.subplots(1, len(DELTA_CONDITIONS), figsize=(4.5 * len(DELTA_CONDITIONS), 4))
if len(DELTA_CONDITIONS) == 1:
    axes = [axes]
for ax, condition in zip(axes, DELTA_CONDITIONS, strict=True):
    result = shift_results[(DELTA_SAMPLE_ID, condition)]
    xy = result.delta_pca_xy
    sc = ax.scatter(xy[:, 0], xy[:, 1], c=result.delta_norm_map.reshape(-1), cmap='magma', s=28)
    ax.set_title(
        f"{condition}
"
        f"PC1={result.metrics['delta_pc1_explained']:.2f}, "
        f"PC1-3={result.metrics['delta_pc3_cumulative']:.2f}"
    )
    ax.set_xlabel('delta PCA-1')
    ax.set_ylabel('delta PCA-2')
    ax.grid(True, alpha=0.2)
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

for condition in DELTA_CONDITIONS:
    result = shift_results[(DELTA_SAMPLE_ID, condition)]
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
    im0 = axes[0].imshow(result.delta_norm_map, cmap='magma')
    axes[0].set_title(f'{condition}
delta norm')
    axes[0].axis('off')
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    im1 = axes[1].imshow(result.delta_cosine_to_mean_map, cmap='coolwarm', vmin=-1, vmax=1)
    axes[1].set_title('cosine to mean delta')
    axes[1].axis('off')
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


## Cell Role: Displacement Vector Field

Top-1 clean patch가 query patch 기준 어느 방향으로 이동했는지 quiver로 본다. position shift는 실제 geometry가 바뀌므로 displacement가 커지는 것이 정상이고, brightness/noise/blur에서 displacement가 커지면 DINO feature identity가 흔들린다는 신호다.


In [ ]:
QUIVER_SAMPLE_ID = SAMPLE_IDS[0]
QUIVER_CONDITIONS = ['brightness_high', 'gaussian_blur_high', 'gaussian_noise_high', 'position_shift_high']

fig, axes = plt.subplots(1, len(QUIVER_CONDITIONS), figsize=(4.5 * len(QUIVER_CONDITIONS), 4))
if len(QUIVER_CONDITIONS) == 1:
    axes = [axes]

for ax, condition in zip(axes, QUIVER_CONDITIONS, strict=True):
    result = match_results[(QUIVER_SAMPLE_ID, condition)]
    height, width = result.grid_shape
    ys, xs = np.meshgrid(np.arange(height), np.arange(width), indexing='ij')
    ax.imshow(result.displacement_norm_map, cmap='magma')
    ax.quiver(
        xs,
        ys,
        result.displacement_x_map,
        result.displacement_y_map,
        color='cyan',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.004,
    )
    ax.set_title(condition)
    ax.set_xlim(-0.5, width - 0.5)
    ax.set_ylim(height - 0.5, -0.5)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()
